# 03 - Baselines, modelos y escalera de mejoras

Objetivo: que se vea la historia competitiva del proyecto, no solo una tabla suelta.

La parte clave que faltaba en el notebook anterior era la escalera `0.52 -> 0.62+`: CNN log-mel, blend con sklearn, full-train, seeds, mejoras estilo Fashion-MNIST y luego arquitecturas cats/dogs. Este notebook resume esa progresion y conecta cada salto con una decision defendible del curso.


In [1]:
from pathlib import Path
import sys

import pandas as pd
try:
    from IPython.display import display
except ModuleNotFoundError:
    display = print

ROOT = Path.cwd()
if ROOT.name == '03_entrenamiento':
    ROOT = ROOT.parent
INVESTIGATION = ROOT / 'investigation'
sys.path.insert(0, str(INVESTIGATION))

decision_matrix = pd.read_csv(ROOT / '03_entrenamiento' / 'decision_matrix.csv')
training_results = pd.read_csv(ROOT / '03_entrenamiento' / 'training_results.csv')
submission_candidates = pd.read_csv(ROOT / '04_final' / 'submission_candidates.csv')

print(ROOT)
print(f'decisions: {len(decision_matrix)}')
print(f'training rows: {len(training_results)}')
print(f'submission candidates: {len(submission_candidates)}')


/home/santig14/fing/taa/2_TallerAA
decisions: 18
training rows: 13
submission candidates: 12


## 1. Escalera de leaderboard

Estos son los hitos que hay que contar en la entrega. No todos son una corrida fresca del notebook: algunos vienen de `investigation/`, porque entrenar todo de nuevo seria caro. Lo importante es que cada salto tiene artefacto, baseline y decision registrados.


In [2]:
impact_ladder = pd.DataFrame([
    {'stage': '#0', 'change': 'Priors de clase', 'private_lb': 0.03749, 'delta_vs_previous': None, 'why_it_matters': 'valida formato, no escucha audio'},
    {'stage': '#1', 'change': 'Sklearn log-mel stats', 'private_lb': 0.32714, 'delta_vs_previous': 0.28965, 'why_it_matters': 'primer modelo con informacion acustica'},
    {'stage': '#5', 'change': 'Regularizacion C=0.01', 'private_lb': 0.37607, 'delta_vs_previous': 0.04893, 'why_it_matters': 'baseline lineal menos sobreajustado'},
    {'stage': '#13', 'change': 'CNN log-mel 128x512', 'private_lb': 0.52257, 'delta_vs_previous': 0.14650, 'why_it_matters': 'preserva estructura tiempo-frecuencia'},
    {'stage': '#17', 'change': 'Blend CNN + sklearn', 'private_lb': 0.56682, 'delta_vs_previous': 0.04425, 'why_it_matters': 'errores complementarios'},
    {'stage': '#30', 'change': 'Full-train + dos seeds CNN', 'private_lb': 0.58612, 'delta_vs_previous': 0.01930, 'why_it_matters': 'mas datos y menos varianza'},
    {'stage': '#37', 'change': 'Fashion-style full-train', 'private_lb': 0.63080, 'delta_vs_previous': 0.04468, 'why_it_matters': 'He, scheduler y cabeza mejoran entrenamiento'},
    {'stage': '#44', 'change': 'Fashion ensemble refinado', 'private_lb': 0.64665, 'delta_vs_previous': 0.01585, 'why_it_matters': 'ReLU/literal/seeds suman diversidad'},
    {'stage': '#51', 'change': 'Catsdogs separable/headsep blend', 'private_lb': 0.65289, 'delta_vs_previous': 0.00624, 'why_it_matters': 'arquitectura separable y transfer aportan poco pero real'},
    {'stage': '2026-06-28', 'change': 'Separable temporal BiGRU blend', 'private_lb': 0.65928, 'delta_vs_previous': 0.00639, 'why_it_matters': 'agrega lectura temporal explicita'},
    {'stage': '2026-06-29', 'change': 'Global-mel + SE + temporal 1024', 'private_lb': 0.67025, 'delta_vs_previous': 0.01097, 'why_it_matters': 'normalizacion global y mayor contexto temporal'},
])
impact_ladder


,stage,change,private_lb,delta_vs_previous,why_it_matters
0,#0,Priors de clase,0.03749,NaN,"valida formato, no escucha audio"
1,#1,Sklearn log-mel stats,0.32714,0.28965,primer modelo con informacion acustica
2,#5,Regularizacion C=0.01,0.37607,0.04893,baseline lineal menos sobreajustado
3,#13,CNN log-mel 128x512,0.52257,0.14650,preserva estructura tiempo-frecuencia
4,#17,Blend CNN + sklearn,0.56682,0.04425,errores complementarios
5,#30,Full-train + dos seeds CNN,0.58612,0.01930,mas datos y menos varianza
6,#37,Fashion-style full-train,0.63080,0.04468,"He, scheduler y cabeza mejoran entrenamiento"
7,#44,Fashion ensemble refinado,0.64665,0.01585,ReLU/literal/seeds suman diversidad
8,#51,Catsdogs separable/headsep blend,0.65289,0.00624,arquitectura separable y transfer aportan poco...
9,2026-06-28,Separable temporal BiGRU blend,0.65928,0.00639,agrega lectura temporal explicita


## 2. La zona que habia que mostrar: `0.52 -> 0.62+`

Esta es la parte que no podia quedar escondida. Ahi se ve que el proyecto no salto magicamente al final: primero la CNN rompio el techo tabular, despues los blends y el full-train consolidaron, y la etapa Fashion llevo el sistema a la zona `0.63/0.64`.


In [3]:
middle_story = impact_ladder[impact_ladder['stage'].isin(['#13', '#17', '#30', '#37', '#44'])].copy()
middle_story['private_lb'] = middle_story['private_lb'].map(lambda x: f'{x:.5f}')
middle_story['delta_vs_previous'] = middle_story['delta_vs_previous'].map(lambda x: f'+{x:.5f}')
middle_story


,stage,change,private_lb,delta_vs_previous,why_it_matters
3,#13,CNN log-mel 128x512,0.52257,+0.14650,preserva estructura tiempo-frecuencia
4,#17,Blend CNN + sklearn,0.56682,+0.04425,errores complementarios
5,#30,Full-train + dos seeds CNN,0.58612,+0.01930,mas datos y menos varianza
6,#37,Fashion-style full-train,0.63080,+0.04468,"He, scheduler y cabeza mejoran entrenamiento"
7,#44,Fashion ensemble refinado,0.64665,+0.01585,ReLU/literal/seeds suman diversidad


Lectura: `#13` es el cambio de representacion/modelo; `#17` prueba que sklearn seguia aportando; `#30` usa todo curated y dos seeds; `#37-#44` incorporan el paquete de entrenamiento inspirado en notebooks del curso: inicializacion, scheduler, cabeza densa, BatchNorm/dropout y diversidad de variantes.


## 3. Corridas frescas vs evidencia historica

Las corridas frescas documentan que el codigo corre y que la direccion se sostiene. La evidencia historica explica los mejores scores, porque esas corridas largas o full-train ya estaban hechas en `investigation/`.


In [4]:
training_results.assign(
    valid_sort=pd.to_numeric(training_results['valid_lwlrap'], errors='coerce')
).sort_values(['source', 'valid_sort'], ascending=[True, False]).drop(columns='valid_sort')


,run,source,model_family,config,epochs,best_epoch,valid_lwlrap,submission_path,decision,notes
12,current85_sep_temporal_full15,fresh_candidate_blend,ensemble,0.85 * current translated final + 0.15 * separ...,NaN,NaN,NaN,investigation/results/submissions/current85_se...,candidate_needs_kaggle_notebook,Local holdout analog improves 0.841179 -> 0.84...
5,cnn_sepres_head256_e24,fresh_run,cnn_logmel,Separable residual CNN + dense head256 + ReLU,24.0,24.0,0.738289,03_entrenamiento/runs/submissions/cnn_sepres_h...,keep,Fresh run confirms final architecture direction
4,cnn_head256_he_plateau_e24,fresh_run,cnn_logmel,Standard CNN + He init + ReduceLROnPlateau + d...,24.0,24.0,0.673265,03_entrenamiento/runs/submissions/cnn_head256_...,keep,Fresh run confirms optimization/head improvement
0,logreg_c001,fresh_run,sklearn_logmel_stats,LogisticRegression OneVsRest C=0.01 basic log-...,NaN,NaN,0.576401,03_entrenamiento/runs/submissions/sklearn_logm...,keep,Fresh run seed42; best sklearn among C=0.01/0....
1,logreg_c002,fresh_run,sklearn_logmel_stats,LogisticRegression OneVsRest C=0.02 basic log-...,NaN,NaN,0.576353,03_entrenamiento/runs/submissions/sklearn_logm...,discard,Almost tied locally but historical Kaggle was ...
2,logreg_c003,fresh_run,sklearn_logmel_stats,LogisticRegression OneVsRest C=0.03 basic log-...,NaN,NaN,0.570624,03_entrenamiento/runs/submissions/sklearn_logm...,discard,Below C=0.01
3,cnn_standard_e12,fresh_run,cnn_logmel,Standard CNN log-mel image 128x512 cosine sche...,12.0,11.0,0.493959,03_entrenamiento/runs/submissions/cnn_standard...,reference,Short fresh run; documents the image-CNN step ...
11,logmel_cnn_separable_temporal_bigru_full_e40_s...,fresh_run_full_train,cnn_temporal,"Full-train separable temporal BiGRU, 40 epochs...",40.0,40.0,NaN,investigation/submissions/logmel_cnn_separable...,candidate,Generated full-train branch for Kaggle-noteboo...
8,logmel_cnn_catsdogs_sepres_head256,historical_investigation,cnn_logmel,Separable residual + dense head256,56.0,56.0,0.807813,investigation/submissions/logmel_cnn_catsdogs_...,keep,Best individual holdout model among documented...
10,logmel_cnn_separable_temporal_bigru,historical_investigation,cnn_temporal,Separable residual backbone + BiGRU temporal head,40.0,40.0,0.797079,investigation/submissions/logmel_cnn_separable...,blend-only,Does not beat headsep individually but improve...


## 4. Matriz de decisiones

No todo lo que se probo entra al final. La matriz separa `keep`, `blend-only`, `discard` y `reference`; eso evita vender como mejora algo que solo subio localmente o que sirve solo como diversidad.


In [5]:
decision_view = decision_matrix[[
    'idea', 'baseline', 'evidence', 'course_support', 'decision', 'project_use', 'caveat'
]].copy()
decision_view


,idea,baseline,evidence,course_support,decision,project_use,caveat
0,priors,#0,Private LB 0.03749; valida formato de submission,submission plumbing,reference,Smoke test de formato,No cuenta como modelo acustico
1,sklearn_logmel_stats,#0,Private LB 0.32714; primer salto grande usando...,log-mel features; multilabel; modelos lineales,keep,Baseline tabular defendible,Colapsa eje temporal
2,sklearn_logmel_c001,#1,Private LB 0.37607; regularizacion mejora cont...,regularizacion; desbalance; validacion,keep,Baseline tabular fuerte y rama de blend,No seguir micro-ajustando C si no mejora LB
3,extended_logmel_stats,#5,Valid local sube pero Private LB baja a 0.36167,feature engineering; validacion,discard,No promover al final,Ejemplo de sobreajuste local
4,cnn_logmel_image,#5,Valid 0.6528 y Private LB 0.52257,CNN; espectrogramas; BatchNorm; dropout,keep,Modelo neural base,Mas costoso y requiere cache log-mel image
5,cnn_sklearn_blend,#13,Private LB 0.56682; errores complementarios,ensembles; promediado de modelos,keep,Primer blend defendible,Pesos deben documentarse como empiricos si vie...
6,full_train_two_seed_cnn,#17,Private LB 0.58612; full curated + dos seeds r...,validacion; full-train final; bagging/ensembles,keep,Entrenamiento final de configuracion elegida,No agregar seeds indefinidamente
7,fashion_head256_training,#30,Private LB 0.64665 acumulado; valid local fuer...,inicializacion; learning-rate schedule; BatchN...,keep,Rama neural principal,Explicar como paquete de entrenamiento/cabeza ...
8,time_reverse_contrast_aug,#34,Valid baja contra head256,data augmentation; regularizacion,discard,No usar por defecto,Augmentations visuales pueden no preservar sem...
9,separable_residual_headsep,#44,Etapa sube a Private LB 0.65289; local 0.80781...,ResNet; Xception; separable conv; residual,keep,Rama final si artefactos estan validados,Incremento final marginal


In [6]:
summary_by_decision = (
    decision_matrix.groupby('decision')
    .size()
    .rename('count')
    .reset_index()
    .sort_values('decision')
)
summary_by_decision


,decision,count
0,blend-only,4
1,discard,4
2,keep,9
3,reference,1


## 5. Ronda nueva desde `investigation/results/`

La carpeta nueva si trajo ideas validas. La normalizacion global por banda mel y la ventana temporal `1024` mejoraron Kaggle; TTA multi-crop y TTA+MC se descartaron.


In [7]:
new_candidates = submission_candidates[
    submission_candidates['candidate'].str.contains('globalmel|f1024|mcdrop|tta|current835|current85', case=False, na=False)
].copy()
new_candidates[['candidate', 'private_score', 'decision', 'notes']].sort_values('private_score', ascending=False, na_position='last')


,candidate,private_score,decision,notes
11,current475_globalmel200_se125_f1024_200,0.67025,selected_final,Best verified Kaggle private score; copied to ...
10,current550_globalmel250_f1024_200,0.66996,validated_candidate,"Strong no-SE blend, close to selected final"
8,current645_globalmel_sep_temporal_full355,0.66561,validated_candidate,Global mel-band normalization candidate
9,current575_globalmel300_se125,0.66519,validated_candidate,SE helps locally but does not beat globalmel a...
4,current85_sep_temporal_full15,NaN,candidate_needs_kaggle_notebook,Holdout analog improves 0.841179 to 0.846742; ...
5,current835_sep_temporal_full165,NaN,candidate_needs_kaggle_notebook,Geron ensemble sweep; local 0.846913 with arit...


## Comandos reproducibles principales

Baseline tabular:

```bash
python investigation/scripts/train_sklearn_variants.py --data-dir data --models logreg_c001 --feature-set basic --seed 42 --test-size 0.2
```

CNN base sobre log-mel:

```bash
python investigation/scripts/train_logmel_cnn.py --data-dir data --n-mels 128 --frames 512 --architecture standard --epochs 35 --seed 42 --test-size 0.2
```

CNN con mejoras de entrenamiento:

```bash
python investigation/scripts/train_logmel_cnn.py --data-dir data --n-mels 128 --frames 512 --initializer he_normal --scheduler plateau --head-hidden 256 --activation relu --epochs 54 --seed 42 --test-size 0.2
```

Rama temporal fuerte:

```bash
python investigation/scripts/train_logmel_cnn.py --data-dir data --architecture separable_temporal_bigru --n-mels 128 --frames 1024 --full-train
```

Los pesos de ensamble finales quedan documentados en `04_final/final_selection.md`.
